In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import json
import statistics
import numpy as np
import math
import torch
import os
import torch.nn as nn

def get_conf_acc(out, dataset_name):
    outs = []
    accs = []
    for sample in out["samples"][dataset_name]:
        resps = [math.exp(x[0]) for x in sample["filtered_resps"]]
        conf = max(resps)/sum(resps)
        outs.append(conf)
        accs.append(sample["acc"])
    return outs, accs

def std_bar(mean, std):
    tmp_mean = np.array(mean)
    tmp_std = np.array(std)
    
    return tmp_mean - tmp_std, tmp_mean + tmp_std



def generate_results_table(model_name, datasets, tmp_path, tmp_path2, tmp_path_unpruned, fa_approaches):
    results = []
    num = 16
    if "gemma" in model_name:
        num = 9

    for dataset in datasets:
        # Unpruned model
        unpruned_row = {"model": f"{model_name}_unpruned", "dataset": dataset}
        out_file = open(tmp_path_unpruned + model_name + ".json", "r")
        out = json.load(out_file)
        out_file.close()
        
        conf_unp, acc = get_conf_acc(out, dataset)

        unpruned_row["accuracy"] = out["results"][dataset]["acc,none"]

        for fa_approach in fa_approaches:
            s_unpruned = []
            c_unpruned = []

            for i in range(3):
                out_file = open(tmp_path_unpruned +"seeds/"+ model_name + "_" + fa_approach + "_res_3d_" + str(i) + ".json", "r")
                out = json.load(out_file)
                out_file.close()

                s_unpruned.append(out[dataset]["suff"])
                c_unpruned.append(out[dataset]["comp"])

            unpruned_row[f"comprehensiveness {fa_approach}"] = statistics.mean(c_unpruned)
            unpruned_row[f"sufficiency {fa_approach}"] = statistics.mean(s_unpruned)

        
        conf = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
        counter = [[], [], [], [], [], [], [], [], [], []]
        conf_counter = [[], [], [], [], [], [], [], [], [], []]

        for j in range(len(conf_unp)):
            for k in range(1, len(conf)):
                if conf_unp[j] < conf[0]:
                    counter[0].append(acc[j])
                    conf_counter[0].append(conf_unp[j])
                    break
                elif conf[k - 1] <= conf_unp[j] < conf[k]:
                    counter[k].append(acc[j])
                    conf_counter[k].append(conf_unp[j])
                    break

        result = [sum(x) / len(x) if len(x) > 0 else 0 for x in counter]
        confid = [sum(x) / len(x) if len(x) > 0 else 0 for x in conf_counter]
        ece = sum([(len(counter[s]) * abs(res - confid[s])) / len(conf_unp) for s, res in enumerate(result)])
        unpruned_row["ECE"] = ece

        results.append(unpruned_row)

        # Pruned models
        for i in range(num):
            pruned_row = {"model": f"{model_name}_pruned_{i + 1}", "dataset": dataset}

            out_file = open(tmp_path2 + model_name + "_" + str(i) + ".json", "r")
            out = json.load(out_file)
            out_file.close()
            
            conf_pruned, acc = get_conf_acc(out, dataset)
            pruned_row["accuracy"] = out["results"][dataset]["acc,none"]

            for fa_approach in fa_approaches:
                s_pruned = []
                c_pruned = []

                for j in range(3):
                    out_file = open(tmp_path + model_name + "_" + str(i) + "_" + fa_approach + "_res_3d_" + str(j) + ".json", "r")
                    out = json.load(out_file)
                    out_file.close()

                    s_pruned.append(out[dataset]["suff"])
                    c_pruned.append(out[dataset]["comp"])

                pruned_row[f"comprehensiveness {fa_approach}"] = statistics.mean(c_pruned)
                pruned_row[f"sufficiency {fa_approach}"] = statistics.mean(s_pruned)

            
            counter = [[], [], [], [], [], [], [], [], [], []]
            conf_counter = [[], [], [], [], [], [], [], [], [], []]

            for j in range(len(conf_pruned)):
                for k in range(1, len(conf)):
                    if conf_pruned[j] < conf[0]:
                        counter[0].append(acc[j])
                        conf_counter[0].append(conf_pruned[j])
                        break
                    elif conf[k - 1] <= conf_pruned[j] < conf[k]:
                        counter[k].append(acc[j])
                        conf_counter[k].append(conf_pruned[j])
                        break

            result = [sum(x) / len(x) if len(x) > 0 else 0 for x in counter]
            confid = [sum(x) / len(x) if len(x) > 0 else 0 for x in conf_counter]
            ece = sum([(len(counter[s]) * abs(res - confid[s])) / len(conf_pruned) for s, res in enumerate(result)])
            pruned_row["ECE"] = ece

            results.append(pruned_row)

    df_results = pd.DataFrame(results)
    return df_results



In [ ]:
datasets = ["arc_easy", "arc_challenge","openbookqa", "boolq", "rte"]
fa_approaches = ["lime", "ks"]
tmp_path = "./results/pruned/seeds/"
tmp_path2 = "./results/pruned/"
tmp_path_unpruned = "./results/unpruned/"
model_name = "meta-llama/Llama-2-7b-hf"

results_table = generate_results_table(model_name, datasets, tmp_path, tmp_path2, tmp_path_unpruned, fa_approaches)
print(results_table)

                                 model   dataset  accuracy  \
0    meta-llama/Llama-2-7b-hf_unpruned  arc_easy  0.763468   
1    meta-llama/Llama-2-7b-hf_pruned_1  arc_easy  0.760522   
2    meta-llama/Llama-2-7b-hf_pruned_2  arc_easy  0.763468   
3    meta-llama/Llama-2-7b-hf_pruned_3  arc_easy  0.759259   
4    meta-llama/Llama-2-7b-hf_pruned_4  arc_easy  0.759680   
..                                 ...       ...       ...   
80  meta-llama/Llama-2-7b-hf_pruned_12       rte  0.595668   
81  meta-llama/Llama-2-7b-hf_pruned_13       rte  0.566787   
82  meta-llama/Llama-2-7b-hf_pruned_14       rte  0.592058   
83  meta-llama/Llama-2-7b-hf_pruned_15       rte  0.606498   
84  meta-llama/Llama-2-7b-hf_pruned_16       rte  0.581227   

    comprehensiveness lime  sufficiency lime  comprehensiveness ks  \
0                 0.233390          0.189856              0.214612   
1                 0.233865          0.189310              0.213489   
2                 0.236075          0.187063 

In [17]:
results_table

,model,dataset,accuracy,comprehensiveness lime,sufficiency lime,comprehensiveness ks,sufficiency ks,ECE
0,meta-llama/Llama-2-7b-hf_unpruned,arc_easy,0.763468,0.233390,0.189856,0.214612,0.211063,0.094339
1,meta-llama/Llama-2-7b-hf_pruned_1,arc_easy,0.760522,0.233865,0.189310,0.213489,0.214334,0.095950
2,meta-llama/Llama-2-7b-hf_pruned_2,arc_easy,0.763468,0.236075,0.187063,0.214504,0.217633,0.091950
3,meta-llama/Llama-2-7b-hf_pruned_3,arc_easy,0.759259,0.230235,0.176713,0.211960,0.212335,0.094928
4,meta-llama/Llama-2-7b-hf_pruned_4,arc_easy,0.759680,0.234677,0.179938,0.207887,0.215142,0.095962
...,...,...,...,...,...,...,...,...
80,meta-llama/Llama-2-7b-hf_pruned_12,rte,0.595668,0.216928,0.187486,0.158230,0.209824,0.074813
81,meta-llama/Llama-2-7b-hf_pruned_13,rte,0.566787,0.181887,0.147253,0.134182,0.212139,0.145688
82,meta-llama/Llama-2-7b-hf_pruned_14,rte,0.592058,0.178197,0.094880,0.115089,0.163395,0.077028
83,meta-llama/Llama-2-7b-hf_pruned_15,rte,0.606498,0.149929,0.070018,0.086147,0.136678,0.098329


In [ ]:

grouped = results_table.groupby("dataset")

latex_tables = ""
for dataset, group in grouped:
    escaped_dataset = dataset.replace("_", "\\_")

    latex_table = f"\\begin{{table}}[ht]\n\\centering\n\\begin{{tabular}}{{lccccccc}}\n"
    latex_table += "\\hline\n"
    latex_table += "Pruned Layers & Accuracy & Comp. Lime & Suff. Lime & Comp. KS & Suff. KS & ECE \\\\\n"
    latex_table += "\\hline\n"
    
    group = group.copy()
    group["pruned_layers"] = group["model"].str.extract(r"pruned_(\d+)", expand=False).fillna("0").astype(int)
    group = group.sort_values("pruned_layers")
    
    for _, row in group.iterrows():
        pruned_layers = row["pruned_layers"]
        accuracy = f"{row['accuracy']:.3f}"
        comp_lime = f"{row['comprehensiveness lime']:.3f}"
        suff_lime = f"{row['sufficiency lime']:.3f}"
        comp_ks = f"{row['comprehensiveness ks']:.3f}"
        suff_ks = f"{row['sufficiency ks']:.3f}"
        ece = f"{row['ECE']:.3f}"
        
        latex_table += f"{pruned_layers} & {accuracy} & {comp_lime} & {suff_lime} & {comp_ks} & {suff_ks} & {ece} \\\\\n"
    
    latex_table += "\\hline\n"
    latex_table += f"\\end{{tabular}}\n\\caption{{Results Table for {escaped_dataset}}}\n\\label{{tab:results_{dataset}}}\n\\end{{table}}\n\n"
    
    latex_tables += latex_table


print(latex_tables)

\begin{table}[ht]
\centering
\begin{tabular}{lccccccc}
\hline
Pruned Layers & Accuracy & Comp. Lime & Suff. Lime & Comp. KS & Suff. KS & ECE \\
\hline
0 & 0.434 & 0.255 & 0.140 & 0.237 & 0.171 & 0.366 \\
1 & 0.430 & 0.253 & 0.140 & 0.233 & 0.171 & 0.365 \\
2 & 0.429 & 0.266 & 0.152 & 0.248 & 0.183 & 0.367 \\
3 & 0.428 & 0.260 & 0.146 & 0.234 & 0.186 & 0.368 \\
4 & 0.428 & 0.258 & 0.151 & 0.231 & 0.186 & 0.367 \\
5 & 0.432 & 0.252 & 0.143 & 0.223 & 0.182 & 0.361 \\
6 & 0.441 & 0.255 & 0.149 & 0.228 & 0.187 & 0.351 \\
7 & 0.434 & 0.249 & 0.161 & 0.222 & 0.189 & 0.356 \\
8 & 0.430 & 0.244 & 0.162 & 0.223 & 0.187 & 0.358 \\
9 & 0.423 & 0.232 & 0.154 & 0.214 & 0.180 & 0.364 \\
10 & 0.419 & 0.217 & 0.153 & 0.207 & 0.175 & 0.369 \\
11 & 0.421 & 0.196 & 0.160 & 0.177 & 0.185 & 0.379 \\
12 & 0.417 & 0.216 & 0.167 & 0.189 & 0.197 & 0.383 \\
13 & 0.410 & 0.197 & 0.167 & 0.180 & 0.190 & 0.391 \\
14 & 0.397 & 0.182 & 0.160 & 0.169 & 0.178 & 0.404 \\
15 & 0.367 & 0.184 & 0.172 & 0.173 & 0.182 & 0.43

In [ ]:
import os
output_dir = "./results/tables"
output_file = os.path.join(output_dir, "res_table_llama.csv")
os.makedirs(output_dir, exist_ok=True)
results_table.to_csv(output_file, index=False)

In [ ]:
datasets = ["arc_easy", "arc_challenge","openbookqa", "boolq", "rte"]
fa_approaches = ["lime", "ks"]
tmp_path = "./results/pruned/seeds/"
tmp_path2 = "./results/pruned/"
tmp_path_unpruned = "./results/unpruned/"
model_name = "meta-llama/Llama-2-7b-hf"

results_table2 = generate_results_table(model_name, datasets, tmp_path, tmp_path2, tmp_path_unpruned, fa_approaches)
print(results_table2)

                                 model   dataset  accuracy  \
0    meta-llama/Llama-2-7b-hf_unpruned  arc_easy  0.763468   
1    meta-llama/Llama-2-7b-hf_pruned_1  arc_easy  0.760522   
2    meta-llama/Llama-2-7b-hf_pruned_2  arc_easy  0.763468   
3    meta-llama/Llama-2-7b-hf_pruned_3  arc_easy  0.759259   
4    meta-llama/Llama-2-7b-hf_pruned_4  arc_easy  0.759680   
..                                 ...       ...       ...   
80  meta-llama/Llama-2-7b-hf_pruned_12       rte  0.595668   
81  meta-llama/Llama-2-7b-hf_pruned_13       rte  0.566787   
82  meta-llama/Llama-2-7b-hf_pruned_14       rte  0.592058   
83  meta-llama/Llama-2-7b-hf_pruned_15       rte  0.606498   
84  meta-llama/Llama-2-7b-hf_pruned_16       rte  0.581227   

    comprehensiveness lime  sufficiency lime  comprehensiveness ks  \
0                 0.233390          0.189856              0.214612   
1                 0.233865          0.189310              0.213489   
2                 0.236075          0.187063 

In [ ]:
grouped = results_table2.groupby("dataset")
latex_tables = ""

for dataset, group in grouped:
    escaped_dataset = dataset.replace("_", "\\_")

    latex_table = f"\\begin{{table}}[ht]\n\\centering\n\\begin{{tabular}}{{lccccccc}}\n"
    latex_table += "\\hline\n"
    latex_table += "Pruned Layers & Accuracy & Comp. Lime & Suff. Lime & Comp. KS & Suff. KS & ECE \\\\\n"
    latex_table += "\\hline\n"

    group = group.copy()
    group["pruned_layers"] = group["model"].str.extract(r"pruned_(\d+)", expand=False).fillna("0").astype(int)
    group = group.sort_values("pruned_layers")
    
    for _, row in group.iterrows():
        pruned_layers = row["pruned_layers"]
        accuracy = f"{row['accuracy']:.3f}"
        comp_lime = f"{row['comprehensiveness lime']:.3f}"
        suff_lime = f"{row['sufficiency lime']:.3f}"
        comp_ks = f"{row['comprehensiveness ks']:.3f}"
        suff_ks = f"{row['sufficiency ks']:.3f}"
        ece = f"{row['ECE']:.3f}"
        
        latex_table += f"{pruned_layers} & {accuracy} & {comp_lime} & {suff_lime} & {comp_ks} & {suff_ks} & {ece} \\\\\n"
    
    latex_table += "\\hline\n"
    latex_table += f"\\end{{tabular}}\n\\caption{{Results Table for {escaped_dataset}}}\n\\label{{tab:results_{dataset}}}\n\\end{{table}}\n\n"
    
    latex_tables += latex_table

print(latex_tables)

\begin{table}[ht]
\centering
\begin{tabular}{lccccccc}
\hline
Pruned Layers & Accuracy & Comp. Lime & Suff. Lime & Comp. KS & Suff. KS & ECE \\
\hline
0 & 0.434 & 0.255 & 0.140 & 0.237 & 0.171 & 0.366 \\
1 & 0.430 & 0.253 & 0.140 & 0.233 & 0.171 & 0.365 \\
2 & 0.429 & 0.266 & 0.152 & 0.248 & 0.183 & 0.367 \\
3 & 0.428 & 0.260 & 0.146 & 0.234 & 0.186 & 0.368 \\
4 & 0.428 & 0.258 & 0.151 & 0.231 & 0.186 & 0.367 \\
5 & 0.432 & 0.252 & 0.143 & 0.223 & 0.182 & 0.361 \\
6 & 0.441 & 0.255 & 0.149 & 0.228 & 0.187 & 0.351 \\
7 & 0.434 & 0.249 & 0.161 & 0.222 & 0.189 & 0.356 \\
8 & 0.430 & 0.244 & 0.162 & 0.223 & 0.187 & 0.358 \\
9 & 0.423 & 0.232 & 0.154 & 0.214 & 0.180 & 0.364 \\
10 & 0.419 & 0.217 & 0.153 & 0.207 & 0.175 & 0.369 \\
11 & 0.421 & 0.196 & 0.160 & 0.177 & 0.185 & 0.379 \\
12 & 0.417 & 0.216 & 0.167 & 0.189 & 0.197 & 0.383 \\
13 & 0.410 & 0.197 & 0.167 & 0.180 & 0.190 & 0.391 \\
14 & 0.397 & 0.182 & 0.160 & 0.169 & 0.178 & 0.404 \\
15 & 0.367 & 0.184 & 0.172 & 0.173 & 0.182 & 0.43

In [ ]:
datasets = ["arc_easy", "arc_challenge","openbookqa", "boolq", "rte"]
fa_approaches = ["lime", "ks"]
tmp_path = "./results/pruned/seeds/"
tmp_path2 = "./results/pruned/"
tmp_path_unpruned = "./results/unpruned/"
model_name = "mistralai/mistral-7b-v0.1"

results_table2 = generate_results_table(model_name, datasets, tmp_path, tmp_path2, tmp_path_unpruned, fa_approaches)
print(results_table2)

                                  model   dataset  accuracy  \
0    mistralai/mistral-7b-v0.1_unpruned  arc_easy  0.808502   
1    mistralai/mistral-7b-v0.1_pruned_1  arc_easy  0.807660   
2    mistralai/mistral-7b-v0.1_pruned_2  arc_easy  0.805556   
3    mistralai/mistral-7b-v0.1_pruned_3  arc_easy  0.803030   
4    mistralai/mistral-7b-v0.1_pruned_4  arc_easy  0.803872   
..                                  ...       ...       ...   
80  mistralai/mistral-7b-v0.1_pruned_12       rte  0.664260   
81  mistralai/mistral-7b-v0.1_pruned_13       rte  0.653430   
82  mistralai/mistral-7b-v0.1_pruned_14       rte  0.664260   
83  mistralai/mistral-7b-v0.1_pruned_15       rte  0.703971   
84  mistralai/mistral-7b-v0.1_pruned_16       rte  0.667870   

    comprehensiveness lime  sufficiency lime  comprehensiveness ks  \
0                 0.265630          0.155031              0.253858   
1                 0.276207          0.159961              0.261745   
2                 0.272008       

In [ ]:
import os
output_dir = "./results/tables"
output_file = os.path.join(output_dir, "res_table_mistral.csv")
os.makedirs(output_dir, exist_ok=True)

results_table2.to_csv(output_file, index=False)

In [ ]:
import pandas as pd

file_path = "./results/tables/res_table_mistral.csv"
#file_path2 = "./results/tables/res_table_llama.csv"

results_table_tmp = pd.read_csv(file_path)

In [ ]:
unpruned = results_table_tmp[~results_table_tmp['model'].str.contains('_pruned_')]

pruned_layers = ["1", "4", "8", "12", "16"]
pruned = results_table_tmp[results_table_tmp['model'].str.contains('_pruned_') & 
                           results_table_tmp['model'].str.extract(r'pruned_(\d+)')[0].astype(str).isin(pruned_layers)]

filtered_results = pd.concat([unpruned, pruned])

filtered_results['drop_in_accuracy'] = filtered_results.apply(
    lambda row: row['accuracy'] - unpruned[unpruned['dataset'] == row['dataset']]['accuracy'].values[0], axis=1
)

summary_table = filtered_results[['model', 'dataset', 'accuracy', 'drop_in_accuracy']]

In [ ]:
def create_latex_table_for_dataset_model(filtered_results, dataset, model_name):
    df = filtered_results[(filtered_results['dataset'] == dataset) & 
                          (filtered_results['model'].str.startswith(model_name))]
    
    
    d_names = {
    "rte":"RTE",
    "arc_challenge":"ARC Challenge",
    "openbookqa":"OpenBookQA",
    "boolq":"BoolQ",
    "arc_easy":"ARC Easy"
    }
    
    def get_pruned_layers(model):
        if "unpruned" in model:
            return "0 (orig)"
        else:
            return model.split("_")[-1]
    
    df = df.copy()
    df["Pruned Layers"] = df["model"].apply(get_pruned_layers)
    df = df.sort_values("Pruned Layers", key=lambda x: x.replace("0 (orig)", "-1").astype(str).astype(int))

    latex = "\\begin{table*}[ht]\n\\centering\n"
    latex += "\\begin{tabular}{lcccccc}\n"
    latex += "\\hline\n"
    latex += "Pruned Layers & Accuracy & Comp. LIME & Suff. LIME & Comp. KS & Suff. KS & ECE \\\\\n"
    latex += "\\hline\n"
    
    for _, row in df.iterrows():
        latex += (
            f"{row['Pruned Layers']} & "
            f"{row['accuracy']:.3f} & "
            f"{row['comprehensiveness lime']:.3f} & "
            f"{row['sufficiency lime']:.3f} & "
            f"{row['comprehensiveness ks']:.3f} & "
            f"{row['sufficiency ks']:.3f} & "
            f"{row['ECE']:.3f} \\\\\n"
        )
    latex += "\\hline\n"
    latex += "\\end{tabular}\n"
   
    if "stra" in model_name:
        model_name_latex = "Mistral"
    else:
        model_name_latex = "Llama-2"
        
    dataset_latex = d_names[dataset]

    latex += f"\\caption{{Results for all measures for {model_name_latex} on {dataset_latex}. Comp. identifies comprehensiveness, Suff. represents sufficiency, KS represents Kernel SHAP, orig is the original unpruned model.}}\n"
    latex += f"\\label{{tab:results_{model_name.replace('/','_').replace('-','_')}_{dataset}}}\n"

    latex += "\\end{table*}"
    return latex

In [ ]:
for i, mo in enumerate(["mistralai/mistral-7b-v0.1", "meta-llama/Llama-2-7b-hf"]):
    file_path = ["./results/tables/res_table_mistral.csv", "./results/tables/res_table_llama.csv"]

    results_table_tmp = pd.read_csv(file_path[i])

    for ds in ["arc_easy", "arc_challenge","openbookqa", "boolq", "rte"]:
        print(create_latex_table_for_dataset_model(results_table_tmp, ds, mo))
        print("\n\n")


\begin{table*}[ht]
\centering
\begin{tabular}{lcccccc}
\hline
Pruned Layers & Accuracy & Comp. LIME & Suff. LIME & Comp. KS & Suff. KS & ECE \\
\hline
0 (orig) & 0.809 & 0.266 & 0.155 & 0.254 & 0.200 & 0.077 \\
1 & 0.808 & 0.276 & 0.160 & 0.262 & 0.204 & 0.078 \\
2 & 0.806 & 0.272 & 0.154 & 0.258 & 0.199 & 0.081 \\
3 & 0.803 & 0.271 & 0.157 & 0.259 & 0.202 & 0.082 \\
4 & 0.804 & 0.278 & 0.160 & 0.266 & 0.206 & 0.083 \\
5 & 0.803 & 0.280 & 0.158 & 0.264 & 0.205 & 0.084 \\
6 & 0.802 & 0.289 & 0.161 & 0.271 & 0.213 & 0.084 \\
7 & 0.798 & 0.284 & 0.163 & 0.264 & 0.209 & 0.089 \\
8 & 0.798 & 0.280 & 0.160 & 0.264 & 0.205 & 0.089 \\
9 & 0.801 & 0.294 & 0.179 & 0.278 & 0.228 & 0.084 \\
10 & 0.796 & 0.294 & 0.173 & 0.275 & 0.220 & 0.087 \\
11 & 0.786 & 0.295 & 0.183 & 0.273 & 0.226 & 0.093 \\
12 & 0.761 & 0.270 & 0.174 & 0.242 & 0.212 & 0.107 \\
13 & 0.721 & 0.264 & 0.180 & 0.240 & 0.207 & 0.120 \\
14 & 0.665 & 0.248 & 0.176 & 0.226 & 0.206 & 0.153 \\
15 & 0.656 & 0.238 & 0.166 & 0.207 & 0.195

In [ ]:
def format_model_name(model):
    if "unpruned" in model:
        return model+" (orig)"
    elif "pruned" in model:
        pruned_layers = model.split("_")[-1]
        return pruned_layers
    return model

summary_table["formatted_model"] = summary_table["model"].apply(format_model_name)
pivot_table = summary_table.pivot(index="formatted_model", columns="dataset", values=["accuracy", "drop_in_accuracy"])
latex_table = "\\begin{table}[ht]\n\\centering\n\\begin{tabular}{l" + "cc" * len(pivot_table.columns.levels[1]) + "}\n"
latex_table += "\\hline\n"

latex_table += "Model & " + " & ".join(
    [f"\\multicolumn{{2}}{{c}}{{{dataset_escaped}}}" for dataset in pivot_table.columns.levels[1]
    for dataset_escaped in [dataset.replace('_', '\\_')]]
) + " \\\\\n"

latex_table += " & " + " & ".join(["Acc & Drop" for _ in pivot_table.columns.levels[1]]) + " \\\\\n"
latex_table += "\\hline\n"

for model, row in pivot_table.iterrows():
    latex_table += model + " & " + " & ".join(
        [
            f"{row[('accuracy', dataset)]:.3f} & {row[('drop_in_accuracy', dataset)]:.3f}"
            if not pd.isna(row[('accuracy', dataset)]) else "- & -"
            for dataset in pivot_table.columns.levels[1]
        ]
    ) + " \\\\\n"

latex_table += "\\hline\n"
latex_table += "\\end{tabular}\n\\caption{Summary of Accuracy and Drop in Accuracy for Different Models and Datasets}\n\\label{tab:summary_table}\n\\end{table}"

print(latex_table)

\begin{table}[ht]
\centering
\begin{tabular}{lcccccccccc}
\hline
Model & \multicolumn{2}{c}{arc\_challenge} & \multicolumn{2}{c}{arc\_easy} & \multicolumn{2}{c}{boolq} & \multicolumn{2}{c}{openbookqa} & \multicolumn{2}{c}{rte} \\
 & Acc & Drop & Acc & Drop & Acc & Drop & Acc & Drop & Acc & Drop \\
\hline
1 & 0.500 & -0.004 & 0.808 & -0.001 & 0.836 & 0.001 & 0.328 & 0.002 & 0.661 & -0.018 \\
12 & 0.464 & -0.040 & 0.761 & -0.048 & 0.772 & -0.064 & 0.282 & -0.044 & 0.664 & -0.014 \\
16 & 0.373 & -0.131 & 0.592 & -0.216 & 0.501 & -0.335 & 0.252 & -0.074 & 0.668 & -0.011 \\
4 & 0.501 & -0.003 & 0.804 & -0.005 & 0.835 & -0.000 & 0.320 & -0.006 & 0.661 & -0.018 \\
8 & 0.484 & -0.020 & 0.798 & -0.011 & 0.830 & -0.006 & 0.328 & 0.002 & 0.675 & -0.004 \\
mistralai/mistral-7b-v0.1_unpruned (orig) & 0.504 & 0.000 & 0.809 & 0.000 & 0.836 & 0.000 & 0.326 & 0.000 & 0.679 & 0.000 \\
\hline
\end{tabular}
\caption{Summary of Accuracy and Drop in Accuracy for Different Models and Datasets}
\label{tab:sum

anonymized path: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_table["formatted_model"] = summary_table["model"].apply(format_model_name)
